# Plan B - Databricks Personal Connections Report

Scans the entire Power BI tenant using the Scanner API and produces
a report of all Databricks datasources with owner contact details.

In [13]:
import json
import time
import pandas as pd
import requests as http_requests

try:
    pbi_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
except Exception:
    pbi_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com/")

HEADERS = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type":  "application/json",
}

PBI_API = "https://api.powerbi.com/v1.0/myorg"

KNOWN_VNET_GATEWAY_IDS = {
    "3e481240-199c-402f-8fe3-ae5ef09b1ab4",
}

print("Setup complete.")

StatementMeta(, e4e88eaa-3651-43fc-aea1-4d802cd24d61, 15, Finished, Available, Finished, False)

Setup complete.


In [2]:
# Fetch capacities
print("Fetching capacities...")
cap_map = {}

try:
    cap_resp = http_requests.get(f"{PBI_API}/admin/capacities", headers=HEADERS)
    if cap_resp.status_code == 200:
        for cap in cap_resp.json().get("value", []):
            cap_map[cap["id"]] = {
                "capacity_name": cap.get("displayName", ""),
                "capacity_sku":  cap.get("sku", ""),
            }
        print(f"{len(cap_map)} capacities loaded.")
    else:
        print(f"Status {cap_resp.status_code}: {cap_resp.text[:300]}")
except Exception as e:
    print(f"Error: {e}")

StatementMeta(, e4e88eaa-3651-43fc-aea1-4d802cd24d61, 4, Finished, Available, Finished, False)

Fetching capacities...
10 capacities loaded.


In [3]:
# Fetch all workspaces
print("Fetching all workspaces...")
all_workspaces = []
top = 5000
skip = 0

while True:
    ws_resp = http_requests.get(
        f"{PBI_API}/admin/groups?$top={top}&$skip={skip}",
        headers=HEADERS,
    )
    if ws_resp.status_code == 429:
        retry = int(ws_resp.headers.get("Retry-After", 30))
        print(f"Rate limited. Sleeping {retry}s...")
        time.sleep(retry)
        continue
    if ws_resp.status_code != 200:
        print(f"Status {ws_resp.status_code}: {ws_resp.text[:300]}")
        break
    batch = ws_resp.json().get("value", [])
    if not batch:
        break
    all_workspaces.extend(batch)
    if len(batch) < top:
        break
    skip += top

ws_map = {}
for ws in all_workspaces:
    ws_map[ws["id"]] = {
        "workspace_name": ws.get("name", ""),
        "capacity_id":    ws.get("capacityId", ""),
    }

print(f"Total workspaces: {len(all_workspaces)}")

StatementMeta(, e4e88eaa-3651-43fc-aea1-4d802cd24d61, 5, Finished, Available, Finished, False)

Fetching all workspaces...
Total workspaces: 17804


In [4]:
# Batch scan using Scanner API
BATCH_SIZE = 100
POLL_INTERVAL = 5
MAX_POLL_TIME = 300

ws_ids = [ws["id"] for ws in all_workspaces]
total_batches = (len(ws_ids) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Scanning {len(ws_ids)} workspaces in {total_batches} batches...")

all_scan_workspaces = []
scan_errors = []

SCAN_URL = (
    f"{PBI_API}/admin/workspaces/getInfo"
    f"?datasourceDetails=True"
    f"&lineage=True"
    f"&datasetSchema=False"
    f"&datasetExpressions=False"
)

for batch_num in range(total_batches):
    start = batch_num * BATCH_SIZE
    end   = start + BATCH_SIZE
    batch_ids = ws_ids[start:end]

    print(f"  Batch {batch_num + 1}/{total_batches} ({len(batch_ids)} workspaces)...")

    scan_body = {"workspaces": batch_ids}

    try:
        scan_resp = http_requests.post(SCAN_URL, headers=HEADERS, json=scan_body)
        if scan_resp.status_code == 429:
            retry = int(scan_resp.headers.get("Retry-After", 60))
            print(f"    Rate limited. Sleeping {retry}s...")
            time.sleep(retry)
            scan_resp = http_requests.post(SCAN_URL, headers=HEADERS, json=scan_body)
        if scan_resp.status_code not in (200, 202):
            print(f"    Failed: {scan_resp.status_code} {scan_resp.text[:300]}")
            scan_errors.append({"batch": batch_num, "error": scan_resp.text[:500]})
            continue
        scan_info = scan_resp.json()
        scan_id = scan_info.get("id", scan_info.get("scanId", ""))
    except Exception as e:
        print(f"    Error: {e}")
        scan_errors.append({"batch": batch_num, "error": str(e)})
        continue

    elapsed = 0
    scan_status = ""
    while elapsed < MAX_POLL_TIME:
        time.sleep(POLL_INTERVAL)
        elapsed += POLL_INTERVAL
        try:
            status_resp = http_requests.get(
                f"{PBI_API}/admin/workspaces/scanStatus/{scan_id}",
                headers=HEADERS,
            )
            if status_resp.status_code == 200:
                scan_status = status_resp.json().get("status", "").lower()
                if scan_status == "succeeded":
                    break
                elif scan_status == "failed":
                    break
            elif status_resp.status_code == 429:
                retry = int(status_resp.headers.get("Retry-After", 15))
                time.sleep(retry)
                elapsed += retry
        except Exception:
            pass

    if scan_status != "succeeded":
        print(f"    Not succeeded (status: {scan_status})")
        scan_errors.append({"batch": batch_num, "status": scan_status})
        continue

    try:
        result_resp = http_requests.get(
            f"{PBI_API}/admin/workspaces/scanResult/{scan_id}",
            headers=HEADERS,
        )
        if result_resp.status_code == 200:
            ws_results = result_resp.json().get("workspaces", [])
            all_scan_workspaces.extend(ws_results)
            print(f"    OK: {len(ws_results)} workspaces.")
        else:
            print(f"    Result failed: {result_resp.status_code}")
    except Exception as e:
        print(f"    Error: {e}")

    time.sleep(2)

print(f"\nScan complete. {len(all_scan_workspaces)} workspaces scanned.")
if scan_errors:
    print(f"{len(scan_errors)} batch errors.")

StatementMeta(, e4e88eaa-3651-43fc-aea1-4d802cd24d61, 6, Finished, Available, Finished, False)

Scanning 17804 workspaces in 179 batches...
  Batch 1/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 2/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 3/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 4/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 5/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 6/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 7/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 8/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 9/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 10/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 11/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 12/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 13/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 14/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 15/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 16/179 (100 workspaces)...
    OK: 100 workspaces.
  Batch 17/179 (100 w

In [6]:
# Diagnostic: show the ACTUAL structure returned so we know where
# datasource details live and how to parse them

if all_scan_workspaces:
    print(f"Workspace-level keys: {sorted(all_scan_workspaces[0].keys())}\n")

    # Show dataset-level keys
    for ws in all_scan_workspaces:
        for ds in ws.get("datasets", []):
            print(f"Dataset-level keys: {sorted(ds.keys())}\n")
            break
        break

    # Search for ANY key containing 'source' or 'instance' anywhere
    def find_keys_recursive(obj, prefix=""):
        found = set()
        if isinstance(obj, dict):
            for k, v in obj.items():
                full = f"{prefix}.{k}" if prefix else k
                if any(term in k.lower() for term in ["source", "instance", "connection", "gateway"]):
                    found.add(full)
                found.update(find_keys_recursive(v, full))
        elif isinstance(obj, list) and obj:
            found.update(find_keys_recursive(obj[0], f"{prefix}[0]"))
        return found

    # Check first 5 workspaces that have datasets
    all_found_keys = set()
    checked = 0
    for ws in all_scan_workspaces:
        if ws.get("datasets"):
            all_found_keys.update(find_keys_recursive(ws))
            checked += 1
            if checked >= 5:
                break

    print(f"Keys containing 'source/instance/connection/gateway':")
    for k in sorted(all_found_keys):
        print(f"  {k}")

    # Show a raw sample of a workspace that mentions 'databricks'
    print(f"\n--- Looking for a workspace with 'databricks' in its data ---")
    for ws in all_scan_workspaces:
        ws_str = json.dumps(ws).lower()
        if "databricks" in ws_str or "azuredatabricks" in ws_str:
            print(f"Found in workspace: {ws.get('name', '')}")
            print(json.dumps(ws, indent=2)[:5000])
            break
    else:
        print("No workspace mentions 'databricks' in scan data.")
        print("Showing first workspace with datasets instead:")
        for ws in all_scan_workspaces:
            if ws.get("datasets"):
                # Show workspace but truncate tables
                ws_copy = dict(ws)
                for ds in ws_copy.get("datasets", []):
                    ds["tables"] = f"({len(ds.get('tables', []))} tables)"
                print(json.dumps(ws_copy, indent=2)[:5000])
                break
else:
    print("No scan results.")

StatementMeta(, e4e88eaa-3651-43fc-aea1-4d802cd24d61, 8, Finished, Available, Finished, False)

Workspace-level keys: ['dashboards', 'dataflows', 'datamarts', 'datasets', 'description', 'id', 'isOnDedicatedCapacity', 'name', 'reports', 'state', 'type']

Dataset-level keys: ['configuredBy', 'configuredById', 'contentProviderType', 'createdDate', 'datasourceUsages', 'id', 'isEffectiveIdentityRequired', 'isEffectiveIdentityRolesRequired', 'name', 'tables', 'tags', 'targetStorageMode']

Keys containing 'source/instance/connection/gateway':
  datasets[0].datasourceUsages
  datasets[0].datasourceUsages[0].datasourceInstanceId

--- Looking for a workspace with 'databricks' in its data ---
Found in workspace: AUDACS_DEV
{
  "id": "270ae2f5-c8f6-4dd2-8c7a-75d051bf67a8",
  "name": "AUDACS_DEV",
  "type": "Workspace",
  "state": "Active",
  "isOnDedicatedCapacity": true,
  "capacityId": "B11E1512-5589-478D-AD89-6544B15A522B",
  "defaultDatasetStorageFormat": "Small",
  "CopyJob": [
    {
      "id": "764f003b-b2ef-43ac-bba1-9092d5ebccca",
      "name": "copyjob1",
      "description": null,

In [7]:
# Parse results - this cell will be updated based on diagnostic output above
# For now it tries ALL known locations for datasource info

print("Parsing scan results...\n")
results = []

for ws_data in all_scan_workspaces:
    ws_id   = ws_data.get("id", "")
    ws_name = ws_data.get("name", "")

    capacity_id = ws_data.get("capacityId", "")
    if not capacity_id:
        capacity_id = ws_map.get(ws_id, {}).get("capacity_id", "")
    cap_info      = cap_map.get(capacity_id, {})
    capacity_name = cap_info.get("capacity_name", "")
    capacity_sku  = cap_info.get("capacity_sku", "")

    # Build datasource instance lookup from ALL possible locations
    ds_instance_map = {}

    # Location 1: workspace.datasourceInstances
    for inst in ws_data.get("datasourceInstances", []):
        inst_id = inst.get("datasourceId", inst.get("datasourceInstanceId", inst.get("id", "")))
        if inst_id:
            ds_instance_map[inst_id] = inst

    # Location 2: workspace.datasets[].datasourceInstances
    for dataset in ws_data.get("datasets", []):
        for inst in dataset.get("datasourceInstances", []):
            inst_id = inst.get("datasourceId", inst.get("datasourceInstanceId", inst.get("id", "")))
            if inst_id:
                ds_instance_map[inst_id] = inst

    # Location 3: workspace.datasets[].datasources
    for dataset in ws_data.get("datasets", []):
        for inst in dataset.get("datasources", []):
            inst_id = inst.get("datasourceId", inst.get("datasourceInstanceId", inst.get("id", "")))
            if inst_id:
                ds_instance_map[inst_id] = inst

    for dataset in ws_data.get("datasets", []):
        dataset_id     = dataset.get("id", "")
        dataset_name   = dataset.get("name", "")
        configured_by  = dataset.get("configuredBy", "")

        usage_ids = [
            u.get("datasourceInstanceId", "")
            for u in dataset.get("datasourceUsages", [])
        ]

        # If we have instance details, use them
        if ds_instance_map:
            for inst_id in usage_ids:
                src = ds_instance_map.get(inst_id, {})
                if not src:
                    continue

                ds_type  = src.get("datasourceType", "")
                conn_raw = src.get("connectionDetails", {})
                conn_str = json.dumps(conn_raw) if conn_raw else "{}"
                gw_id    = src.get("gatewayId", "")

                is_dbx = False
                if ds_type.lower() in ("databricks", "azuredatabricks"):
                    is_dbx = True
                elif ds_type.lower() == "extension":
                    if "databricks" in conn_str.lower():
                        is_dbx = True
                if not is_dbx:
                    continue

                if not gw_id:
                    gw_class = "No Gateway"
                elif gw_id in KNOWN_VNET_GATEWAY_IDS:
                    gw_class = "VNet Gateway"
                else:
                    gw_class = "Personal Cloud Gateway"

                host_val      = conn_raw.get("host", conn_raw.get("server", conn_raw.get("path", "")))
                http_path_val = conn_raw.get("httpPath", conn_raw.get("database", ""))
                if isinstance(host_val, str) and host_val.strip().startswith("{"):
                    try:
                        parsed = json.loads(host_val)
                        host_val = parsed.get("host", parsed.get("server", host_val))
                        if not http_path_val:
                            http_path_val = parsed.get("httpPath", "")
                    except Exception:
                        pass

                results.append({
                    "capacity_id":        capacity_id,
                    "capacity_name":      capacity_name,
                    "capacity_sku":       capacity_sku,
                    "workspace_id":       ws_id,
                    "workspace_name":     ws_name,
                    "dataset_id":         dataset_id,
                    "dataset_name":       dataset_name,
                    "datasource_id":      inst_id,
                    "datasource_type":    ds_type,
                    "gateway_id":         gw_id,
                    "gateway_class":      gw_class,
                    "host":               host_val,
                    "http_path":          http_path_val,
                    "connection_details": conn_str,
                    "owner_name":         configured_by,
                    "owner_email":        configured_by,
                })
        else:
            # No instance details available - record the IDs anyway
            for inst_id in usage_ids:
                results.append({
                    "capacity_id":        capacity_id,
                    "capacity_name":      capacity_name,
                    "capacity_sku":       capacity_sku,
                    "workspace_id":       ws_id,
                    "workspace_name":     ws_name,
                    "dataset_id":         dataset_id,
                    "dataset_name":       dataset_name,
                    "datasource_id":      inst_id,
                    "datasource_type":    "(scan has no details)",
                    "gateway_id":         "",
                    "gateway_class":      "(unknown)",
                    "host":               "",
                    "http_path":          "",
                    "connection_details": "",
                    "owner_name":         configured_by,
                    "owner_email":        configured_by,
                })

print(f"Total rows: {len(results)}")
print(f"Rows with details: {sum(1 for r in results if r['datasource_type'] != '(scan has no details)')}")
print(f"Rows without details: {sum(1 for r in results if r['datasource_type'] == '(scan has no details)')}")

StatementMeta(, e4e88eaa-3651-43fc-aea1-4d802cd24d61, 9, Finished, Available, Finished, False)

Parsing scan results...

Total rows: 3098
Rows with details: 0
Rows without details: 3098


In [8]:
# If scan didn't return datasource details, enrich using the
# per-dataset admin API but ONLY for datasets that have
# datasourceUsages (targeted calls, not full tenant scan)

needs_enrichment = [r for r in results if r["datasource_type"] == "(scan has no details)"]

if needs_enrichment:
    print(f"Scan didn't return datasource details.")
    print(f"Enriching {len(needs_enrichment)} datasource references via targeted API calls...")
    print(f"(Only calling datasets that have datasourceUsages, with rate limit handling)\n")

    # Get unique dataset IDs that need enrichment
    datasets_to_enrich = {}
    for r in needs_enrichment:
        ds_id = r["dataset_id"]
        if ds_id not in datasets_to_enrich:
            datasets_to_enrich[ds_id] = r["workspace_id"]

    print(f"Unique datasets to query: {len(datasets_to_enrich)}")

    # Fetch datasources for each dataset
    datasource_cache = {}  # datasource_instance_id -> datasource details
    api_calls = 0

    for idx, (dataset_id, workspace_id) in enumerate(datasets_to_enrich.items()):
        if (idx + 1) % 50 == 0:
            print(f"  Progress: {idx + 1}/{len(datasets_to_enrich)}")

        try:
            resp = http_requests.get(
                f"{PBI_API}/admin/datasets/{dataset_id}/datasources",
                headers=HEADERS,
            )
            api_calls += 1

            if resp.status_code == 429:
                retry = int(resp.headers.get("Retry-After", 30))
                print(f"  Rate limited at call {api_calls}. Sleeping {retry}s...")
                time.sleep(retry)
                resp = http_requests.get(
                    f"{PBI_API}/admin/datasets/{dataset_id}/datasources",
                    headers=HEADERS,
                )
                api_calls += 1

            if resp.status_code == 200:
                for src in resp.json().get("value", []):
                    src_inst_id = src.get("datasourceId", "")
                    if src_inst_id:
                        datasource_cache[src_inst_id] = src
        except Exception:
            pass

        # Throttle: 1 call per 0.3s = ~200/min
        if api_calls % 10 == 0:
            time.sleep(1)

    print(f"\nDone. {api_calls} API calls, {len(datasource_cache)} datasources cached.")

    # Update results with enriched data
    enriched_results = []
    for r in results:
        if r["datasource_type"] == "(scan has no details)":
            src = datasource_cache.get(r["datasource_id"], {})
            if src:
                ds_type  = src.get("datasourceType", "")
                conn_raw = src.get("connectionDetails", {})
                conn_str = json.dumps(conn_raw) if conn_raw else "{}"
                gw_id    = src.get("gatewayId", "")

                is_dbx = False
                if ds_type.lower() in ("databricks", "azuredatabricks"):
                    is_dbx = True
                elif ds_type.lower() == "extension":
                    if "databricks" in conn_str.lower():
                        is_dbx = True

                if not is_dbx:
                    continue  # skip non-databricks

                if not gw_id:
                    gw_class = "No Gateway"
                elif gw_id in KNOWN_VNET_GATEWAY_IDS:
                    gw_class = "VNet Gateway"
                else:
                    gw_class = "Personal Cloud Gateway"

                host_val      = conn_raw.get("host", conn_raw.get("server", conn_raw.get("path", "")))
                http_path_val = conn_raw.get("httpPath", conn_raw.get("database", ""))
                if isinstance(host_val, str) and host_val.strip().startswith("{"):
                    try:
                        parsed = json.loads(host_val)
                        host_val = parsed.get("host", parsed.get("server", host_val))
                        if not http_path_val:
                            http_path_val = parsed.get("httpPath", "")
                    except Exception:
                        pass

                r["datasource_type"]    = ds_type
                r["gateway_id"]         = gw_id
                r["gateway_class"]      = gw_class
                r["host"]               = host_val
                r["http_path"]          = http_path_val
                r["connection_details"] = conn_str
                enriched_results.append(r)
            # else: no data for this datasource, skip
        else:
            enriched_results.append(r)

    results = enriched_results
    print(f"\nFinal Databricks rows: {len(results)}")
else:
    print("Scan returned datasource details directly. No enrichment needed.")

StatementMeta(, e4e88eaa-3651-43fc-aea1-4d802cd24d61, 10, Finished, Available, Finished, False)

Scan didn't return datasource details.
Enriching 3098 datasource references via targeted API calls...
(Only calling datasets that have datasourceUsages, with rate limit handling)

Unique datasets to query: 1658
  Progress: 50/1658
  Progress: 100/1658
  Progress: 150/1658
  Progress: 200/1658
  Progress: 250/1658
  Progress: 300/1658
  Rate limited at call 301. Sleeping 3467s...
  Progress: 350/1658
  Progress: 400/1658
  Progress: 450/1658
  Progress: 500/1658
  Progress: 550/1658
  Progress: 600/1658
  Progress: 650/1658
  Progress: 700/1658
  Progress: 750/1658
  Progress: 800/1658
  Progress: 850/1658
  Progress: 900/1658
  Progress: 950/1658
  Progress: 1000/1658
  Progress: 1050/1658
  Progress: 1100/1658
  Progress: 1150/1658
  Progress: 1200/1658
  Progress: 1250/1658
  Progress: 1300/1658
  Progress: 1350/1658
  Progress: 1400/1658
  Progress: 1450/1658
  Progress: 1500/1658
  Progress: 1550/1658
  Progress: 1600/1658
  Progress: 1650/1658

Done. 1659 API calls, 411 datasource

In [9]:
# Build report and export
df_planb = pd.DataFrame(results)

if df_planb.empty:
    print("No Databricks datasources found.")
else:
    df_planb = df_planb.sort_values(
        by=["gateway_class", "owner_email", "capacity_name", "workspace_name"],
        ascending=[False, True, True, True],
    ).reset_index(drop=True)

    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)

    print(f"\nBy gateway type:")
    print(df_planb.groupby("gateway_class").size().to_string())

    print(f"\nBy capacity:")
    print(df_planb.groupby(["capacity_name", "capacity_sku", "gateway_class"]).size().to_string())

    df_personal = df_planb[df_planb["gateway_class"] != "VNet Gateway"].copy()
    if not df_personal.empty:
        print(f"\nBy owner (non-VNet only):")
        print(df_personal.groupby(["owner_email", "gateway_class"]).size().to_string())
        print(f"\nDatasources NOT on VNet: {len(df_personal)}")
        print(f"Unique owners to contact: {df_personal['owner_email'].nunique()}")
    else:
        print("\nAll on VNet gateway!")

    display_cols = [
        "capacity_name", "workspace_name", "dataset_name",
        "host", "http_path", "gateway_class",
        "owner_name", "owner_email",
    ]
    existing_cols = [c for c in display_cols if c in df_planb.columns]
    display(df_planb[existing_cols])

    try:
        csv_all = "/lakehouse/default/Files/all_databricks_connections.csv"
        df_planb.to_csv(csv_all, index=False)
        print(f"\nFull report: {csv_all}")
    except Exception:
        csv_all = "/tmp/all_databricks_connections.csv"
        df_planb.to_csv(csv_all, index=False)
        print(f"\nFull report: {csv_all}")

    if not df_personal.empty:
        try:
            csv_p = "/lakehouse/default/Files/personal_connections.csv"
            df_personal.to_csv(csv_p, index=False)
            print(f"Personal only: {csv_p}")
        except Exception:
            csv_p = "/tmp/personal_connections.csv"
            df_personal.to_csv(csv_p, index=False)
            print(f"Personal only: {csv_p}")

StatementMeta(, e4e88eaa-3651-43fc-aea1-4d802cd24d61, 11, Finished, Available, Finished, False)

No Databricks datasources found.


In [10]:
# Email templates
if not df_personal.empty:
    print("=" * 70)
    print("EMAIL TEMPLATES")
    print("=" * 70)

    for owner_email, grp in df_personal.groupby("owner_email"):
        datasets_list = "\n".join([
            f"   - {row['dataset_name']}"
            f"  (workspace: {row['workspace_name']}"
            f", capacity: {row['capacity_name']})"
            for _, row in grp.iterrows()
        ])
        hosts = grp["host"].unique()
        paths = grp["http_path"].unique()

        print(f"\n{'─' * 60}")
        print(f"To: {owner_email}")
        print(f"Subject: Action Required - Migrate Databricks connection")
        print(f"")
        print(f"Hi {grp.iloc[0]['owner_name']},")
        print(f"")
        print(f"These datasets need migration to VNet Data Gateway:")
        print(f"")
        print(datasets_list)
        print(f"")
        print(f"Host(s): {', '.join(str(h) for h in hosts)}")
        print(f"Path(s): {', '.join(str(p) for p in paths)}")
        print(f"")
        print(f"Steps:")
        print(f"  1. Power BI > Settings > Manage connections and gateways")
        print(f"  2. Select VNet Data Gateway")
        print(f"  3. Add data source > Databricks > enter host + HTTP path")
        print(f"  4. Sign in with OAuth2")
        print(f"  5. Dataset settings > Gateway > select VNet datasource")
        print(f"  6. Refresh dataset")
        print(f"{'─' * 60}")
else:
    print("No personal connections to report.")

StatementMeta(, e4e88eaa-3651-43fc-aea1-4d802cd24d61, 12, Finished, Available, Finished, False)

NameError: name 'df_personal' is not defined

In [18]:
# FINAL REPORT - Built from Fabric Connections API (all_connections + dbx_connections already in memory)

rows = []
for conn in dbx_connections:
    conn_id   = conn.get("id", "")
    name      = conn.get("displayName", "") or ""
    conn_type = conn.get("connectivityType", "")
    gw_id     = conn.get("gatewayId", "")
    details   = conn.get("connectionDetails", {})
    cred      = conn.get("credentialDetails", {})
    path_raw  = details.get("path", "")
    ds_type   = details.get("type", "")
    cred_type = cred.get("credentialType", "")
    privacy   = conn.get("privacyLevel", "")
    allow_gw  = conn.get("allowConnectionUsageInGateway", False)

    # Parse host and httpPath from path
    host_val = ""
    http_path_val = ""
    if path_raw.strip().startswith("{"):
        try:
            parsed = json.loads(path_raw)
            host_val = parsed.get("host", "") or ""
            http_path_val = parsed.get("httpPath", "") or ""
        except Exception:
            host_val = path_raw
    elif path_raw.startswith("http"):
        host_val = path_raw
    else:
        host_val = path_raw

    # Classify
    if conn_type == "VirtualNetworkGateway":
        gw_class = "✅ VNet Gateway"
    elif conn_type == "PersonalCloud":
        gw_class = "⚠️ Personal Cloud"
    elif conn_type == "ShareableCloud":
        gw_class = "☁️ Shareable Cloud"
    else:
        gw_class = conn_type

    rows.append({
        "connection_id":    conn_id,
        "connection_name":  name,
        "connectivity_type": conn_type,
        "gateway_class":    gw_class,
        "gateway_id":       gw_id,
        "databricks_type":  ds_type,
        "host":             host_val,
        "http_path":        http_path_val,
        "credential_type":  cred_type,
        "privacy_level":    privacy,
        "allow_in_gateway": allow_gw,
        "skip_test":        cred.get("skipTestConnection", False),
        "path_raw":         path_raw,
    })

df_dbx = pd.DataFrame(rows)

# ── Summary ──
print("=" * 70)
print(f"📊 ALL DATABRICKS CONNECTIONS IN TENANT: {len(df_dbx)}")
print("=" * 70)

print(f"\nBy connectivity type:")
print(df_dbx.groupby("gateway_class").size().to_string())

print(f"\nBy Databricks type:")
print(df_dbx.groupby("databricks_type").size().to_string())

print(f"\nBy credential type:")
print(df_dbx.groupby("credential_type").size().to_string())

# ── Separate VNet vs non-VNet ──
df_vnet = df_dbx[df_dbx["connectivity_type"] == "VirtualNetworkGateway"]
df_not_vnet = df_dbx[df_dbx["connectivity_type"] != "VirtualNetworkGateway"]

print(f"\n✅ Already on VNet Gateway: {len(df_vnet)}")
print(f"⚠️  NOT on VNet Gateway:    {len(df_not_vnet)}")

# ── Display all ──
display_cols = [
    "connection_name", "gateway_class", "databricks_type",
    "host", "http_path", "credential_type", "allow_in_gateway",
]
print("\n")
display(df_dbx[display_cols])

# ── Now cross-reference with scan data to find which datasets use these connections ──
# Build a set of all Databricks connection IDs
dbx_conn_ids = set(df_dbx["connection_id"].tolist())

# Search scan results for datasets referencing these connections
print("\n" + "=" * 70)
print("📋 DATASETS USING DATABRICKS CONNECTIONS")
print("=" * 70)

dataset_rows = []
for ws_data in all_scan_workspaces:
    ws_id   = ws_data.get("id", "")
    ws_name = ws_data.get("name", "")
    
    capacity_id = ws_data.get("capacityId", "")
    if not capacity_id:
        capacity_id = ws_map.get(ws_id, {}).get("capacity_id", "")
    cap_info      = cap_map.get(capacity_id, {})
    capacity_name = cap_info.get("capacity_name", "")
    capacity_sku  = cap_info.get("capacity_sku", "")

    for dataset in ws_data.get("datasets", []):
        dataset_id    = dataset.get("id", "")
        dataset_name  = dataset.get("name", "")
        configured_by = dataset.get("configuredBy", "")
        
        usages = dataset.get("datasourceUsages", [])
        for usage in usages:
            inst_id = usage.get("datasourceInstanceId", "")
            if inst_id in dbx_conn_ids:
                conn_info = df_dbx[df_dbx["connection_id"] == inst_id].iloc[0]
                dataset_rows.append({
                    "capacity_id":       capacity_id,
                    "capacity_name":     capacity_name,
                    "capacity_sku":      capacity_sku,
                    "workspace_id":      ws_id,
                    "workspace_name":    ws_name,
                    "dataset_id":        dataset_id,
                    "dataset_name":      dataset_name,
                    "connection_id":     inst_id,
                    "connection_name":   conn_info["connection_name"],
                    "gateway_class":     conn_info["gateway_class"],
                    "databricks_type":   conn_info["databricks_type"],
                    "host":              conn_info["host"],
                    "http_path":         conn_info["http_path"],
                    "credential_type":   conn_info["credential_type"],
                    "owner_name":        configured_by,
                    "owner_email":       configured_by,
                })

df_datasets = pd.DataFrame(dataset_rows)

if not df_datasets.empty:
    df_datasets = df_datasets.sort_values(
        by=["gateway_class", "owner_email", "workspace_name"],
        ascending=[False, True, True],
    ).reset_index(drop=True)

    df_personal_ds = df_datasets[df_datasets["gateway_class"] != "✅ VNet Gateway"].copy()

    print(f"\nDatasets linked to Databricks connections: {len(df_datasets)}")
    print(f"  On VNet:     {len(df_datasets) - len(df_personal_ds)}")
    print(f"  NOT on VNet: {len(df_personal_ds)}")
    if not df_personal_ds.empty:
        print(f"  Unique owners to contact: {df_personal_ds['owner_email'].nunique()}")

    ds_display = [
        "capacity_name", "workspace_name", "dataset_name",
        "connection_name", "gateway_class", "host", "http_path",
        "credential_type", "owner_name", "owner_email",
    ]
    display(df_datasets[[c for c in ds_display if c in df_datasets.columns]])
else:
    print("\nNo datasets reference these Databricks connections via datasourceUsages.")
    print("The connections exist but may be used by Dataflows, Pipelines, or Notebooks.")

# ── Export ──
try:
    csv1 = "/lakehouse/default/Files/databricks_connections.csv"
    df_dbx.to_csv(csv1, index=False)
    print(f"\n💾 Connections: {csv1}")
except Exception:
    csv1 = "/tmp/databricks_connections.csv"
    df_dbx.to_csv(csv1, index=False)
    print(f"\n💾 Connections: {csv1}")

if not df_datasets.empty:
    try:
        csv2 = "/lakehouse/default/Files/databricks_dataset_connections.csv"
        df_datasets.to_csv(csv2, index=False)
        print(f"💾 Dataset mapping: {csv2}")
    except Exception:
        csv2 = "/tmp/databricks_dataset_connections.csv"
        df_datasets.to_csv(csv2, index=False)
        print(f"💾 Dataset mapping: {csv2}")

# ── Connections NOT on VNet (the action items) ──
if not df_not_vnet.empty:
    print("\n" + "=" * 70)
    print("⚠️  CONNECTIONS NEEDING MIGRATION TO VNET GATEWAY:")
    print("=" * 70)
    for _, r in df_not_vnet.iterrows():
        print(f"\n  {r['connection_name'] or '(unnamed)'}")
        print(f"    ID:    {r['connection_id']}")
        print(f"    Type:  {r['gateway_class']} / {r['databricks_type']}")
        print(f"    Host:  {r['host']}")
        print(f"    Path:  {r['http_path']}")
        print(f"    Cred:  {r['credential_type']}")

StatementMeta(, e4e88eaa-3651-43fc-aea1-4d802cd24d61, 20, Finished, Available, Finished, False)

📊 ALL DATABRICKS CONNECTIONS IN TENANT: 13

By connectivity type:
gateway_class
☁️ Shareable Cloud    9
⚠️ Personal Cloud     1
✅ VNet Gateway        3

By Databricks type:
databricks_type
AzureDatabricksWorkspace    5
Databricks                  7
DatabricksMultiCloud        1

By credential type:
credential_type
Key                 8
OAuth2              1
ServicePrincipal    4

✅ Already on VNet Gateway: 3
⚠️  NOT on VNet Gateway:    10




SynapseWidget(Synapse.DataFrame, ad2b35cd-efb7-41fb-868f-f3cb231b140a)


📋 DATASETS USING DATABRICKS CONNECTIONS

Datasets linked to Databricks connections: 5
  On VNet:     4
  NOT on VNet: 1
  Unique owners to contact: 1


SynapseWidget(Synapse.DataFrame, c9ad8ade-212b-4449-8931-2e9b40a0e628)


💾 Connections: /tmp/databricks_connections.csv
💾 Dataset mapping: /tmp/databricks_dataset_connections.csv

⚠️  CONNECTIONS NEEDING MIGRATION TO VNET GATEWAY:

  AzDatabricks_AUDACS_DEV
    ID:    b9d8e3ff-aa46-4047-b709-8d1ac2585660
    Type:  ☁️ Shareable Cloud / DatabricksMultiCloud
    Host:  adb-2906548049715187.7.azuredatabricks.net
    Path:  sql/protocolv1/o/2906548049715187/0707-213040-spmq1kl2
    Cred:  Key

  AzDatabricks_Pipeline_AUDACS_DEV carfonseca
    ID:    90cd7be0-1047-4934-82b2-ff77e04a732d
    Type:  ☁️ Shareable Cloud / Databricks
    Host:  adb-2906548049715187.7.azuredatabricks.net
    Path:  sql/protocolv1/o/2906548049715187/0707-213040-spmq1kl2
    Cred:  Key

  AzDatabricks_Workspace_AUDACS_DEV 31c1830f da93231a
    ID:    15979a27-be43-4449-a0e7-304c78f02454
    Type:  ☁️ Shareable Cloud / AzureDatabricksWorkspace
    Host:  https://adb-2906548049715187.7.azuredatabricks.net
    Path:  
    Cred:  Key

  {"host":"adb-2906548049715187.7.azuredatabricks.net",